# Provenance you can re-check: receipts and bit-identical replay

An answer from a model is only as trustworthy as your ability to *audit* it later. mixle 0.7.0 adds
two provenance primitives that make an answer offline-re-verifiable — by anyone, without re-running
the original computation from scratch or trusting that it was done right:

* **`ExecutionTrace`** records the exact tool calls (name, arguments, and RNG **seed**) behind an
  answer, so the whole thing **replays bit-identically** — and any tampering with the record is
  detected by re-execution.
* **`Receipt`** binds four independently-checkable claims into one artifact — a decision **ledger**,
  an execution **trace**, a **calibration** record, and **provenance** — and `verify_receipt` checks
  each one offline.

Everything here runs offline with two trivial "tools" (a text op and a seeded RNG draw).

## 1. A trace that replays bit-identically

A tool step records its name, its arguments, and — for anything stochastic — the seed it ran under.
`record_step` captures the result; `replay` re-runs the tools from the recorded args/seed and must
reproduce the *exact same bytes*.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from mixle.task.replay import ExecutionTrace, TraceStep, record_step, replay, is_bit_identical_replay, diff

def draw_normal(n, seed):            # a stochastic tool -- reproducible only if the seed is recorded
    return np.random.RandomState(seed).normal(size=n).tolist()
def uppercase(text):                 # a deterministic tool
    return text.upper()
TOOLS = {'draw_normal': draw_normal, 'uppercase': uppercase}

step1 = record_step(TOOLS, 'uppercase',   {'text': 'ship it'})
step2 = record_step(TOOLS, 'draw_normal', {'n': 3}, seed=42)
trace = ExecutionTrace(request='make a decision', steps=[step1, step2])

print('recorded stochastic result:', trace.steps[1].result)
print('replays bit-identically   :', is_bit_identical_replay(trace, TOOLS))

recorded stochastic result: [0.4967141530112327, -0.13826430117118466, 0.6476885381006925]
replays bit-identically   : True


Because the trace serializes to JSON and back, it can be stored next to the answer and re-verified
by someone who only has the recorded tools — no access to the original session required.

In [2]:
restored = ExecutionTrace.from_json(trace.to_json())
print('round-trips through JSON  :', restored.dumps() == trace.dumps())
print('restored trace re-verifies:', is_bit_identical_replay(restored, TOOLS))

round-trips through JSON  : True
restored trace re-verifies: True


## 2. Tampering is caught by re-execution

The trace is not a log you have to trust — it is a *claim you can refute*. Change the recorded seed
(while keeping the old result) and `diff` flags exactly which step no longer reproduces.

In [3]:
tampered = ExecutionTrace(request='make a decision', steps=[
    step1,
    TraceStep(tool='draw_normal', args={'n': 3}, seed=99, result=step2.result),  # seed changed, result stale
])
mismatches = diff(tampered, replay(tampered, TOOLS))
print('steps that fail to reproduce (index, tool):', mismatches)

steps that fail to reproduce (index, tool): [(1, 'draw_normal')]


## 3. A full receipt: ledger + trace + calibration + provenance

A `Receipt` binds four independently-checkable claims. `verify_receipt` checks each **offline** and
reports `pass` / `fail` / `absent` per claim — absent claims are honest, not failures.

In [4]:
from mixle.inference.receipt import Receipt, verify_receipt
from mixle.inference.explain import explain
from mixle.stats import CategoricalDistribution, CompositeDistribution, GaussianDistribution

model = CompositeDistribution((CategoricalDistribution({'approve': 0.9, 'deny': 0.1}),
                               GaussianDistribution(0.0, 1.0)))
receipt = Receipt(
    answer='approve',
    produced_by='student-v1',
    ledger=explain(model, ('approve', 0.5)),        # the additive decision-margin ledger
    trace=trace,                                     # the replayable tool trace from section 1
    calibration={'alpha': 0.1, 'qhat': 0.83},        # a named conformal calibration
    provenance={'source_id': 'corpus-42'},           # where the answer came from
)
report = verify_receipt(receipt, tools=TOOLS)
print('receipt verifies offline:', report.passed)
for check, outcome in report.checks.items():
    print('  %-22s %s' % (check, outcome))

receipt verifies offline: True
  ledger_exact           pass
  trace_replayable       pass
  calibration_named      pass
  provenance_present     pass


And the point of "verifiable" — corrupt any one claim and verification fails on *that* claim, while
the others still stand. Here we break the trace's recorded result:

In [5]:
import copy
bad = copy.deepcopy(trace)
bad.steps[1].result = [9.9, 9.9, 9.9]                # not what the seed actually produces
broken = Receipt(answer='approve', trace=bad, provenance={'source_id': 'corpus-42'})
report = verify_receipt(broken, tools=TOOLS)
print('tampered-trace receipt verifies:', report.passed)
print('  trace_replayable :', report.checks['trace_replayable'])
print('  provenance_present:', report.checks['provenance_present'])

tampered-trace receipt verifies: False
  trace_replayable : fail
  provenance_present: pass


## What 0.7.0 gives you here

Provenance stops being "trust me" and becomes *checkable*:

* an **`ExecutionTrace`** replays **bit-identically** from recorded args + seeds, and `diff` pinpoints
  any step that has been altered — the record refutes itself if it's wrong;
* a **`Receipt`** binds ledger + trace + calibration + provenance into one artifact that
  **`verify_receipt` checks offline**, claim by claim, with a tampered claim failing in isolation.

This is what lets a distilled student, an agent, or a pipeline hand you an answer you can *audit
later* — not just accept.